# Exploratory Data Analysis (EDA) - Merged Listings Dataset

This notebook explores the uncleaned merged dataset at `data/processed/main/merged.csv`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
RANDOM_STATE = 42

In [ ]:
candidate_paths = [
    Path('data/processed/main/merged.csv'),
    Path('../data/processed/main/merged.csv'),
    Path.cwd() / 'data/processed/main/merged.csv',
]

csv_path = None
for p in candidate_paths:
    if p.exists():
        csv_path = p
        break

if csv_path is None:
    raise FileNotFoundError('Could not locate merged.csv in expected locations.')

df = pd.read_csv(csv_path)
print(f'Loaded: {csv_path.resolve()}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
df.head()

In [ ]:
print('DataFrame info:')
print('-' * 80)
df.info()

print('\nBasic checks:')
print('-' * 80)
print(f'Duplicated rows: {df.duplicated().sum():,}')
print(f'Total missing cells: {df.isna().sum().sum():,}')
print(f'Rows with at least one missing value: {df.isna().any(axis=1).sum():,}')

missing_pct_by_row = (df.isna().mean(axis=1) * 100)
print(f'Median missing % per row: {missing_pct_by_row.median():.2f}%')
print(f'95th percentile missing % per row: {missing_pct_by_row.quantile(0.95):.2f}%')

## Missingness and Data Quality Checks

In [ ]:
missing_summary = (
    df.isna()
    .sum()
    .rename('missing_count')
    .to_frame()
    .assign(missing_pct=lambda x: (x['missing_count'] / len(df) * 100).round(2))
    .sort_values('missing_count', ascending=False)
)

missing_summary.head(20)

plt.figure(figsize=(12, 6))
plot_missing = missing_summary[missing_summary['missing_count'] > 0].head(20)
if not plot_missing.empty:
    sns.barplot(
        data=plot_missing.reset_index(),
        x='missing_pct',
        y='index',
        color='steelblue'
    )
    plt.title('Top 20 Columns by Missing Percentage')
    plt.xlabel('Missing (%)')
    plt.ylabel('Column')
else:
    plt.text(0.5, 0.5, 'No missing values found', ha='center', va='center')
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
df_plot = df.copy()
converted_to_numeric = []
converted_to_bool = []

for col in df_plot.columns:
    series = df_plot[col]
    if series.dtype != 'object':
        continue

    non_null = series.dropna().astype(str).str.strip()
    if non_null.empty:
        continue

    upper_values = set(non_null.str.upper().unique())
    if upper_values.issubset({'TRUE', 'FALSE'}):
        mapped = series.astype(str).str.strip().str.upper().map({'TRUE': True, 'FALSE': False})
        mapped[series.isna()] = np.nan
        df_plot[col] = mapped
        converted_to_bool.append(col)
        continue

    numeric_candidate = pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False).str.strip(),
        errors='coerce'
    )
    if numeric_candidate.notna().mean() >= 0.85:
        df_plot[col] = numeric_candidate
        converted_to_numeric.append(col)

print(f'Converted to numeric for EDA: {len(converted_to_numeric)} columns')
print(converted_to_numeric[:20])
print(f'Converted to boolean for EDA: {len(converted_to_bool)} columns')
print(converted_to_bool)

df_plot.select_dtypes(include='number').describe().T.head(15)

In [ ]:
categorical_focus = [
    col for col in ['city', 'country', 'geographic_zone', 'room_type', 'listing_type', 'cancellation_policy']
    if col in df_plot.columns
]

if categorical_focus:
    n_cols = 2
    n_rows = int(np.ceil(len(categorical_focus) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4.5 * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for ax, col in zip(axes, categorical_focus):
        top_counts = df_plot[col].astype(str).value_counts(dropna=False).head(12)
        sns.barplot(
            x=top_counts.values,
            y=top_counts.index,
            ax=ax,
            color='seagreen'
        )
        ax.set_title(f'Top Categories: {col}')
        ax.set_xlabel('Count')
        ax.set_ylabel('')

    for ax in axes[len(categorical_focus):]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

for col in categorical_focus:
    print(f'\nTop values for {col}:')
    display(df_plot[col].value_counts(dropna=False).head(10).to_frame('count'))

In [ ]:
numeric_cols = df_plot.select_dtypes(include='number').columns.tolist()

priority_numeric = [
    'ttm_revenue', 'ttm_avg_rate', 'ttm_occupancy', 'ttm_revpar',
    'l90d_revenue', 'l90d_avg_rate', 'l90d_occupancy', 'l90d_revpar',
    'distance_from_city_center', 'rating_overall', 'num_reviews',
    'guests', 'bedrooms', 'beds', 'baths'
]

hist_cols = [c for c in priority_numeric if c in numeric_cols]
if len(hist_cols) < 12:
    extras = [c for c in numeric_cols if c not in hist_cols]
    hist_cols.extend(extras[: 12 - len(hist_cols)])

hist_cols = hist_cols[:12]
n_cols = 3
n_rows = int(np.ceil(len(hist_cols) / n_cols)) if hist_cols else 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4.5 * n_rows))
axes = np.atleast_1d(axes).flatten()

for ax, col in zip(axes, hist_cols):
    sns.histplot(df_plot[col].dropna(), bins=40, kde=True, ax=ax, color='teal')
    ax.set_title(f'Histogram: {col}')

for ax in axes[len(hist_cols):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

## Correlation and Relationship Analysis

In [ ]:
corr_priority = [
    'ttm_revenue', 'ttm_avg_rate', 'ttm_occupancy', 'ttm_adjusted_occupancy',
    'ttm_revpar', 'ttm_adjusted_revpar', 'l90d_revenue', 'l90d_avg_rate',
    'l90d_occupancy', 'l90d_adjusted_occupancy', 'l90d_revpar',
    'distance_from_city_center', 'rating_overall', 'num_reviews',
    'guests', 'bedrooms', 'beds', 'baths'
]

corr_cols = [c for c in corr_priority if c in df_plot.columns and c in numeric_cols]

if len(corr_cols) >= 2:
    corr_matrix = df_plot[corr_cols].corr(numeric_only=True)

    plt.figure(figsize=(14, 10))
    sns.heatmap(
        corr_matrix,
        cmap='RdBu_r',
        center=0,
        vmin=-1,
        vmax=1,
        annot=False,
        square=True,
        cbar_kws={'shrink': 0.8}
    )
    plt.title('Correlation Heatmap (Selected Numeric Features)')
    plt.tight_layout()
    plt.show()

    corr_pairs = (
        corr_matrix.where(~np.eye(corr_matrix.shape[0], dtype=bool))
        .stack()
        .rename('correlation')
        .reset_index()
        .rename(columns={'level_0': 'feature_1', 'level_1': 'feature_2'})
    )
    corr_pairs['abs_corr'] = corr_pairs['correlation'].abs()
    top_corr = corr_pairs.sort_values('abs_corr', ascending=False).drop_duplicates(subset=['abs_corr']).head(12)

    print('Top absolute correlations (excluding self-correlation):')
    display(top_corr[['feature_1', 'feature_2', 'correlation']])
else:
    print('Not enough numeric columns available for correlation analysis.')

In [ ]:
scatter_pairs = [
    ('ttm_avg_rate', 'ttm_occupancy'),
    ('distance_from_city_center', 'ttm_revenue'),
    ('l90d_avg_rate', 'l90d_occupancy'),
    ('num_reviews', 'rating_overall'),
]

available_pairs = [
    (x, y) for x, y in scatter_pairs
    if x in df_plot.columns and y in df_plot.columns
]

plot_df = df_plot.copy()
for x, y in available_pairs:
    plot_df = plot_df[plot_df[x].notna() & plot_df[y].notna()]

if len(plot_df) > 8000:
    plot_df = plot_df.sample(8000, random_state=RANDOM_STATE)

if available_pairs:
    n_cols = 2
    n_rows = int(np.ceil(len(available_pairs) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
    axes = np.atleast_1d(axes).flatten()

    hue_col = 'geographic_zone' if 'geographic_zone' in plot_df.columns else None

    for ax, (x_col, y_col) in zip(axes, available_pairs):
        sns.scatterplot(
            data=plot_df,
            x=x_col,
            y=y_col,
            hue=hue_col,
            alpha=0.35,
            s=25,
            ax=ax,
            legend=False
        )
        ax.set_title(f'Scatter: {y_col} vs {x_col}')

    for ax in axes[len(available_pairs):]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()
else:
    print('No predefined scatterplot pairs found in current data.')

In [ ]:
key_metrics = [
    c for c in ['ttm_revenue', 'ttm_avg_rate', 'ttm_occupancy', 'l90d_revenue', 'l90d_avg_rate', 'l90d_occupancy']
    if c in df_plot.columns
]

if key_metrics:
    all_key_missing = df_plot[key_metrics].isna().all(axis=1)
    print(f'Rows with all key performance metrics missing: {all_key_missing.sum():,} / {len(df_plot):,}')

outlier_cols = [c for c in ['ttm_revenue', 'l90d_revenue', 'ttm_avg_rate', 'l90d_avg_rate', 'distance_from_city_center'] if c in numeric_cols]
if outlier_cols:
    n_cols = 3
    n_rows = int(np.ceil(len(outlier_cols) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4.5 * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for ax, col in zip(axes, outlier_cols):
        sns.boxplot(x=df_plot[col], ax=ax, color='lightcoral')
        ax.set_title(f'Boxplot: {col}')

    for ax in axes[len(outlier_cols):]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()